In [1]:
# https://www.aluracursos.com/blog/langgraph-que-es-como-usarlo-y-sus-funcionalidades
# Instalar bibliotecas necesarias
!pip install -q -U langgraph langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 2.8 MB/s eta 0:00:00


In [2]:
!pip show langgraph

Name: langgraph
Version: 1.2.11
Summary: Building stateful, multi-actor applications with LLMs
Home-page: https://docs.langchain.com/oss/python/langgraph/overview
Author: 
Author-email: 
License: 
Location: /usr/local/lib/python3.13/dist-packages
Requires: langchain-core, langgraph-checkpoint, langgraph-prebuilt, langgraph-sdk, pydantic, xxhash
Required-by: langchain


In [3]:
import os
# Permite usar los secrets
from google.colab import userdata
# Obteniendo la llave de OpenAI de forma segura
os.environ['OPENAI_API_KEY']=userdata.get('OPENAI_API_KEY')

In [4]:
# Proyecto LangGraph
# Importaciones principales
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from typing import TypedDict
# Definición del estado global
class Estado(TypedDict):
    pregunta: str
    respuesta: str
# Iniciando el modelo LLM
modelo = ChatOpenAI(model="gpt-4o-mini")
# Nodo 1: Recibe y muestra la pregunta
def nodo_recibir_pregunta(state: Estado):
    print("🔹 Pregunta recibida:", state["pregunta"])
    return state
# Nodo 2: Genera la respuesta vía LLM y la guarda en el estado
def nodo_generar_respuesta(state: Estado):
    pregunta = state["pregunta"]
    respuesta = modelo.invoke(pregunta)  # Llamada al LLM
    state["respuesta"] = respuesta.content
    print("🔹 Respuesta generada con éxito.")
    return state
# Construyendo el grafo
grafo = StateGraph(Estado)
grafo.add_node("recibir_pregunta", nodo_recibir_pregunta)
grafo.add_node("generar_respuesta", nodo_generar_respuesta)
grafo.add_edge("recibir_pregunta", "generar_respuesta")
grafo.add_edge("generar_respuesta", END)
# Definiendo el inicio del flujo
grafo.set_entry_point("recibir_pregunta")
# Compila el grafo
app = grafo.compile()

In [6]:
# Ejecutando el grafo
estado_inicial = {
    "pregunta": "Explica qué es LangGraph de forma sencilla.",
    "respuesta": ""
}
resultado = app.invoke(estado_inicial)
print("\n===== Respuesta del agente =====")
print(resultado["respuesta"])

🔹 Pregunta recibida: Explica qué es LangGraph de forma sencilla.
🔹 Respuesta generada con éxito.

===== Respuesta del agente =====
LangGraph es una herramienta que combina el procesamiento del lenguaje natural con el uso de grafos, permitiendo organizar y analizar información de manera más efectiva. Imagina que es como un mapa que ayuda a conectar ideas y conceptos de forma visual y lógica. Esto puede ser útil para comprender relaciones, encontrar información relevante y hacer consultas más complejas en grandes cantidades de datos. En resumen, LangGraph facilita la manera en que interactuamos con la información escrita, ayudándonos a captar mejor el contexto y las conexiones entre diferentes datos.
